In [1]:
import numpy as np
import scvelo as scv
import scanpy as sc
import pandas as pd
import topovelo as tpv
import matplotlib.pyplot as plt

In [8]:
FIG_DIR=''
MODEL_DIR = ''
DATA_DIR = ''

# Load preprocessed data

In [14]:
adata = sc.read('')

# TopoVelo

In [15]:
from sklearn.neighbors import NearestNeighbors
X_pos = adata.obsm['X_spatial']
nn = NearestNeighbors(n_neighbors=16)
nn.fit(X_pos)
adata.obsp['spatial_graph'] = nn.kneighbors_graph()

In [17]:
figure_path = f'{FIG_DIR}/gcn'
model_path = f'{MODEL_DIR}/gcn'

vae = tpv.VAE(adata, 
              tmax=20, 
              dim_z=10, 
              device='cuda:0',
              graph_decoder=True,
              attention=False,
              reverse_gene_mode=False)
config = {
    'batch_size':32
}
genes = ['Foxp2', 'Gnas', 'Npm1']
vae.train(adata,
          adata.obsp['spatial_graph'],
          "X_spatial",
          config=config,
          plot=False,
          gene_plot=genes,
          figure_path=figure_path,
          embed='spatial')
vae.save_model(model_path, 'encoder', 'decoder')
vae.save_anndata(adata, 'gcn', DATA_DIR, file_name="adata_out.h5ad")

Estimating ODE parameters...


  0%|          | 0/606 [00:00<?, ?it/s]

  0%|          | 0/606 [00:00<?, ?it/s]

Reinitialize the regular ODE parameters based on estimated global latent time.


  0%|          | 0/606 [00:00<?, ?it/s]

*********               Creating a Graph Dataset              *********
*********                      Finished.                      *********
--------------------------- Train a TopoVelo ---------------------------
*********                 Creating optimizers                 *********
*********                      Finished.                      *********
*********                    Start training                   *********
*********                      Stage  1                       *********
*********       Stage 1: Early Stop Triggered at epoch 222.       *********
Summary: 
Train ELBO = 781.419
Test ELBO = 739.535
Total Time =   0 h :  1 m : 17 s

*********                      Stage  2                       *********


Calculating KNN:   0%|          | 0/1030 [00:00<?, ?it/s]

Stage 2: Early Stop Triggered at round 11.
Final: Train ELBO = 1410.320,	Test ELBO = 1327.361
*********              Finished. Total Time =   0 h :  2 m : 40 s             *********


## Graph Attention

In [19]:
figure_path = f'{FIG_DIR}/gat'
model_path = f'{MODEL_DIR}/gat'

vae = tpv.VAE(adata, 
              tmax=20, 
              dim_z=10, 
              device='cuda:0',
              graph_decoder=True,
              attention=True,
              reverse_gene_mode=False)
config = {
    'batch_size':32
}
genes = ['Foxp2', 'Gnas', 'Npm1']
vae.train(adata,
          adata.obsp['spatial_graph'],
          "X_spatial",
          config=config,
          plot=False,
          gene_plot=genes,
          figure_path=figure_path,
          embed='spatial')
vae.save_model(model_path, 'encoder_gat', 'decoder_gat')
vae.save_anndata(adata, 'gat', DATA_DIR, file_name="adata_out.h5ad")

Estimating ODE parameters...


  0%|          | 0/606 [00:00<?, ?it/s]

  0%|          | 0/606 [00:00<?, ?it/s]

Reinitialize the regular ODE parameters based on estimated global latent time.


  0%|          | 0/606 [00:00<?, ?it/s]

*********               Creating a Graph Dataset              *********
*********                      Finished.                      *********
--------------------------- Train a TopoVelo ---------------------------
*********                 Creating optimizers                 *********
*********                      Finished.                      *********
*********                    Start training                   *********
*********                      Stage  1                       *********
*********       Stage 1: Early Stop Triggered at epoch 213.       *********
Summary: 
Train ELBO = 782.367
Test ELBO = 728.358
Total Time =   0 h :  3 m : 34 s

*********                      Stage  2                       *********


Calculating KNN:   0%|          | 0/1030 [00:00<?, ?it/s]

Stage 2: Early Stop Triggered at round 14.
Final: Train ELBO = 1373.684,	Test ELBO = 1294.039
*********              Finished. Total Time =   0 h :  7 m : 40 s             *********


# Informative Time Prior

In [20]:
tpv.model.model_util.get_spatial_tprior(adata, 20, q=0.95)

In [ ]:
tpv.plotting.set_dpi(100)
res = tpv.post_analysis(
    adata,
    'gut',
    ['TopoVelo (GCN)', 'TopoVelo (GAT)'],
    ['gcn', 'gat'],
    compute_metrics=False,
    spatial_velocity_graph=True,
    genes=['Foxp2', 'Gnas', 'Npm1'],
    grid_size=(1, 1),
    plot_type=['time', 'cell velocity'],
)